# PatchTST

PatchTST is a Transformer for time series. It cuts each series into fixed-length
patches (like tokens), applies self-attention across them, and uses RevIN
normalization. It is a global, univariate model (past sales only).

WMAE is measured on the real validation rows (from `get_real_validation`), so it is
directly comparable to the tree models. Experiments are logged to the shared DagsHub.

Note: PatchTST is the heaviest model here; the retrain step can take a while on CPU.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.models.patchtst_pipeline import build_patchtst
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Shared DagsHub tracking.
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("PatchTST_Training")

# Load the model config.
with open("configs/patchtst_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-09 20:30:07,793	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-09 20:30:08,250	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/09 20:30:12 INFO mlflow.tracking.fluent: Experiment with name 'PatchTST_Training' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

train: (421570, 5)


## Run 1: Data Preparation

In [4]:
with mlflow.start_run(run_name="PatchTST_Data_Prep"):
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, split_date)
    real_valid = get_real_validation(train_raw, stores, features, split_date)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", split_date)
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_param("real_valid_rows", int(len(real_valid)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon:", horizon,
          "| real valid rows:", len(real_valid))

series: 3331 | horizon: 43 | real valid rows: 127438
🏃 View run PatchTST_Data_Prep at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6/runs/a0aaea966f0a4ef080614bc903312ce6
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6


## WMAE on real validation rows

In [5]:
def evaluate_on_real(forecast_df, real_valid, model_col="PatchTST"):
    merged = real_valid.merge(
        forecast_df[["unique_id", "ds", model_col]], on=["unique_id", "ds"], how="inner",
    )
    preds = merged[model_col].clip(lower=0)  # sales can't be negative
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2: Training

Train PatchTST from the config and score it on the real validation rows.
(Earlier exploration: a lighter baseline gave ~3,438 WMAE on the gap-filled set; this
tuned config with a longer lookback and finer patches won.)

In [6]:
with mlflow.start_run(run_name="PatchTST_Training"):
    mlflow.log_params(params)

    nf = build_patchtst(horizon, params, freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_on_real(fcst, real_valid, "PatchTST")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"PatchTST WMAE (real validation): {wmae:,.2f}")

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 545 K  | train
-----------------------------------------------------------
545 K     Trainable params
3         Non-trainable params
545 K     Total params
2.180     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Epoch 0:  96%|█████████▌| 100/104 [03:26<00:08,  0.48it/s, v_num=3, train_loss_step=4.03e+3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  92%|█████████▏| 96/104 [03:22<00:16,  0.47it/s, v_num=3, train_loss_step=2.73e+3, train_loss_epoch=3.48e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  88%|████████▊ | 92/104 [03:40<00:28,  0.42it/s, v_num=3, train_loss_step=2.17e+3, train_loss_epoch=3.12e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  85%|████████▍ | 88/104 [03:19<00:36,  0.44it/s, v_num=3, train_loss_step=3.04e+3, train_loss_epoch=2.87e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  81%|████████  | 84/104 [03:03<00:43,  0.46it/s, v_num=3, train_loss_step=2.56e+3, train_loss_epoch=2.75e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |    

`Trainer.fit` stopped: `max_steps=800` reached.


Epoch 7: 100%|██████████| 72/72 [02:51<00:00,  0.42it/s, v_num=3, train_loss_step=2.94e+3, train_loss_epoch=2.53e+3]


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 104/104 [00:02<00:00, 39.53it/s]
PatchTST WMAE (real validation): 2,508.67
🏃 View run PatchTST_Training at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6/runs/7cb92e0336d04412b3ebd8a609a57468
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6


## Run 3: Best Pipeline (retrain on all data, save)

Retrain on the full history and save the model. This is the slow step on CPU.

In [7]:
with mlflow.start_run(run_name="PatchTST_Best_Pipeline"):
    mlflow.log_params(params)

    nf_best = build_patchtst(horizon, params, freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])  # full history

    save_path = "saved_models/patchtst_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="patchtst_nf")
    print("Saved PatchTST model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 545 K  | train
-----------------------------------------------------------
545 K     Trainable params
3         Non-trainable params
545 K     Total params
2.180     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Epoch 0:  95%|█████████▌| 100/105 [03:24<00:10,  0.49it/s, v_num=5, train_loss_step=2.64e+3]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  90%|█████████ | 95/105 [03:18<00:20,  0.48it/s, v_num=5, train_loss_step=1.95e+3, train_loss_epoch=3.3e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  86%|████████▌ | 90/105 [03:17<00:32,  0.46it/s, v_num=5, train_loss_step=2.8e+3, train_loss_epoch=2.69e+3]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  81%|████████  | 85/105 [03:07<00:44,  0.45it/s, v_num=5, train_loss_step=1.42e+3, train_loss_epoch=2.52e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  76%|███████▌  | 80/105 [02:43<00:51,  0.49it/s, v_num=5, train_loss_step=1.58e+3, train_loss_epoch=2.39e+3] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |     

`Trainer.fit` stopped: `max_steps=800` reached.


Epoch 7: 100%|██████████| 65/65 [02:24<00:00,  0.45it/s, v_num=5, train_loss_step=1.76e+3, train_loss_epoch=2.25e+3]
Saved PatchTST model to saved_models/patchtst_nf
🏃 View run PatchTST_Best_Pipeline at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6/runs/dfe91d49ac6b4858b9bf98b4a926224c
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/6


## Notes

- WMAE here is on the real validation rows, so it is comparable to the tree models.
- Params live in `configs/patchtst_config.yaml`; the model is built in
  `src/models/patchtst_pipeline.py`.
- PatchTST is univariate in this library, so it uses past sales only.